# Part 4: DistilBERT

Author: Kevin

This is model number four in our comparison. It's a pretrained transformer that I finetuned on the same split everyone in the team used.

Parts 1 and 2 already showed the classes are easy to tell apart just from trigger words like `raped`, `fgm` and `beats`, so the scores are already really high and there isn't much room left to beat them. I picked a transformer for a different reason. It splits words into smaller pieces, so even a misspelled or unseen word still means something to it. It also comes already trained on a huge pile of text, so it isn't starting from scratch like the BiLSTM did with only 27,755 tweets.

In [ ]:
!pip install -q transformers
!git clone -q https://github.com/Ebi-Tech/NLP-and-Language-Technologies.git
%cd NLP-and-Language-Technologies

In [ ]:
import re, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report

from src import metrics as M

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
hardware = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"
print(device, hardware)

## Step 1: Load the shared split

The split file only has IDs in it, so I join it with the cleaned data file to get the real text and labels. If any row ends up without text then something went wrong, so I check that too.

In [ ]:
split = pd.read_csv("splits/train_val_test_split.csv")
text = pd.read_csv("data/processed/train_cleaned.csv")

df = split.merge(text[["Tweet_ID", "tweet", "label_id"]], on="Tweet_ID", how="left")

print(df["split"].value_counts().to_dict())
print("missing text:", df["tweet"].isna().sum())

## Step 2: Only light cleaning

The other models train on `tweet_clean`, which is lowercased and tidied up. I use the raw `tweet` column and only take out URLs and @handles.

The reason is that the tokenizer was pretrained on normal text, so it expects text that still looks normal. If I strip punctuation or stopwords I'd be throwing away stuff the model already knows how to use. `distilbert-base-uncased` lowercases by itself anyway, so we're still even with the other models on that.

In [ ]:
def clean(t):
    t = re.sub(r"http\S+|@\w+", " ", str(t))
    return re.sub(r"\s+", " ", t).strip()

df["text"] = df["tweet"].map(clean)
df[["tweet", "text"]].head(3)

## Step 3: Picking a sequence length

Part 1 found tweets almost never go past 68 words. But this tokenizer breaks rare words into pieces, so one word can turn into three or four tokens. That means the token count is bigger than the word count, and I should check the real numbers before picking a length, otherwise I could be cutting off half of most tweets without noticing.

In [ ]:
MODEL = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)

lens = [len(tok(t)["input_ids"]) for t in df["text"].sample(3000, random_state=42)]
print("median, 90th, 99th:", np.percentile(lens, [50, 90, 99]).round(1))
print("max:", max(lens))

## Step 4: Turn the text into numbers

Each tweet becomes a fixed block of 128 token ids. The attention mask just tells the model which of those are real words and which are padding. The BiLSTM used 70 word slots, so 128 token slots covers about the same amount of text.

In [ ]:
MAX_LEN = 128

def to_dataset(rows):
    enc = tok(list(rows["text"]), truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
    return TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(rows["label_id"].values))

train_rows = df[df["split"] == "train"]
val_rows = df[df["split"] == "val"]
test_rows = df[df["split"] == "test"]

train_ds = to_dataset(train_rows)
val_ds = to_dataset(val_rows)
test_ds = to_dataset(test_rows)

y_val = val_rows["label_id"].values
y_test = test_rows["label_id"].values

print(len(train_ds), len(val_ds), len(test_ds))

## Step 5: Training

This is normal fine-tuning. Load the pretrained weights, put a 5-class layer on top, and train the whole thing with a small learning rate so we don't wreck what it already knows.

`weighted` turns class weights on or off. I left it as a switch instead of hardcoding it because that's the one experiment I run below.

In [ ]:
def predict(model, ds, bs=64):
    model.eval()
    out = []
    with torch.no_grad():
        for ids, mask, _ in DataLoader(ds, batch_size=bs):
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            out.append(logits.cpu())
    return torch.cat(out).argmax(1).numpy()


def train(weighted, epochs=3, lr=2e-5, bs=32):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=5).to(device)

    w = None
    if weighted:
        y = train_rows["label_id"].values
        w = compute_class_weight("balanced", classes=np.arange(5), y=y)
        w = torch.tensor(w, dtype=torch.float, device=device)

    loss_fn = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    loader = DataLoader(train_ds, batch_size=bs, shuffle=True)

    hist = []
    start = time.time()
    for ep in range(epochs):
        model.train()
        total = 0
        for ids, mask, y in loader:
            opt.zero_grad()
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            loss = loss_fn(logits, y.to(device))
            loss.backward()
            opt.step()
            total += loss.item()
        f1 = f1_score(y_val, predict(model, val_ds), average="macro", zero_division=0)
        hist.append({"epoch": ep + 1, "train_loss": total / len(loader), "val_macro_f1": f1})
        print(hist[-1])

    return model, pd.DataFrame(hist), (time.time() - start) / 60

## Step 6: Two versions

The experiment is class weights on versus off. That's the same choice Part 2 made for the BiLSTM, so the two line up.

It matters here because the rarest class only has 188 tweets in the whole dataset. Without weights the model can get a nice accuracy while basically ignoring that class. Running both means I can point at a number instead of just assuming weights help.

In [ ]:
model_w, hist_w, time_w = train(weighted=True)

In [ ]:
model_u, hist_u, time_u = train(weighted=False)

## Step 7: Compare on validation

I compare the two on validation only and keep whichever did better. Test doesn't get touched until after that choice is made.

In [ ]:
f1_w = hist_w["val_macro_f1"].iloc[-1]
f1_u = hist_u["val_macro_f1"].iloc[-1]
print(f"with class weights: {f1_w:.4f}")
print(f"without class weights: {f1_u:.4f}")

if f1_w >= f1_u:
    best, best_time, config = model_w, time_w, "class weights"
else:
    best, best_time, config = model_u, time_u, "no class weights"

print("kept:", config)
print(classification_report(y_val, predict(best, val_ds), target_names=M.LABEL_ORDER, zero_division=0))

## Step 8: Learning curves

Training loss shows if the model is still getting better. Validation macro F1 shows if that's real learning or just memorizing. I plot both versions on the same chart so the effect of the class weights is easy to see.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))

for h, name in [(hist_w, "class weights"), (hist_u, "no class weights")]:
    ax[0].plot(h["epoch"], h["train_loss"], marker="o", label=name)
    ax[1].plot(h["epoch"], h["val_macro_f1"], marker="o", label=name)

ax[0].set_title("Training loss")
ax[1].set_title("Validation macro F1")
for a in ax:
    a.set_xlabel("epoch")
    a.set_xticks(hist_w["epoch"])
    a.legend()

plt.tight_layout()
plt.savefig("reports/figures/distilbert_learning_curve.png", dpi=150)
plt.show()

## Step 9: Score it on test, once

One run on the test set. I use the shared `src/metrics.py` so my numbers come out in the same format as everyone else's.

In [ ]:
test_preds = predict(best, test_ds)
res = M.compute_metrics(y_test, test_preds)

print(M.format_metrics_report(res, "DistilBERT"))
M.save_metrics(res, "DistilBERT", "test")
M.plot_confusion_matrix(res["confusion_matrix"], "DistilBERT")
plt.show()

## Step 10: Save the predictions and my row

I save the prediction for every single test tweet, not just the summary numbers. That way the comparison later, including the no trigger word check, can be done without retraining anything.

Nothing here overwrites a file someone else owns. My comparison row goes into its own file instead of getting appended to `results/model_comparison.csv`, so the shared table only gets built once, later on.

In [ ]:
pd.DataFrame({
    "Tweet_ID": test_rows["Tweet_ID"].values,
    "true_id": y_test,
    "pred_id": test_preds,
}).to_csv("results/preds_distilbert_test.csv", index=False)

row = {"model": "DistilBERT", "split": "test", "accuracy": res["accuracy"], "macro_f1": res["macro_f1"]}
for label in M.LABEL_ORDER:
    row[f"f1_{label}"] = round(res["per_class_report"][label]["f1-score"], 3)
row["hardware"] = hardware
row["training_time_min"] = round(best_time, 1)
row["notes"] = f"distilbert-base-uncased, 3 epochs, lr 2e-5, max_len {MAX_LEN}, {config} selected on validation macro F1"

comp = pd.DataFrame([row])
comp.to_csv("results/model_comparison_distilbert.csv", index=False)
comp

## Notes

This notebook only makes new files, it doesn't edit anyone else's:

- `reports/results/distilbert_test.json`
- `reports/figures/distilbert_learning_curve.png`
- `reports/figures/confusion_matrix_distilbert.png`
- `results/preds_distilbert_test.csv`
- `results/model_comparison_distilbert.csv`

`training_time_min` is something the other parts don't report yet, so I keep it in my own file instead of adding a column to the shared table.

On Colab these get written into the cloned copy, not into GitHub. So download those five files and commit them on the `part_4` branch.